# Python UDFs and Pandas operations in PySpark

Run from top to bottom in a WSL Python kernel with PySpark and Java configured. These examples target Spark 3.5.x and Python 3.10/3.11, matching the Day43 environment. Each notebook creates or reuses a local SparkSession and is self-contained.

A **user-defined function (UDF)** supplies custom logic to Spark SQL or DataFrame expressions. Start with built-in Spark functions; introduce a UDF when the required logic is not readily expressible with them.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import udf, col
from pyspark.sql.types import LongType, DoubleType

spark = (SparkSession.builder.master("local[2]")
         .appName("udf_examples")
         .config("spark.sql.shuffle.partitions", "2")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


## Pandas dependencies
The Pandas examples also require `pandas` and `pyarrow` in the notebook's WSL Python environment. For the Spark 3.5 teaching environment, install these before starting the kernel:

```bash
python -m pip install pandas==2.2.3 pyarrow==17.0.0
```

Restart the kernel after installation. These packages belong in the Linux environment used by the kernel, not only in Windows Python.


## Wrap a Python function and register a SQL name
A UDF extends Spark with custom logic. Use the UDF object directly in DataFrame expressions; register it only when you want to call it by name in SQL. Temporary UDF registrations belong to the SparkSession, not to a notebook file. Two notebooks sharing a session can therefore see the same registration.

This example squares a value. Spark already provides arithmetic and `power`; a UDF is used here to demonstrate the mechanism. `LongType` has a finite 64-bit range, so very large squares need a separate overflow policy.


In [ ]:
def power(n):
    # Spark null values arrive as None in this ordinary Python UDF.
    return None if n is None else n * n

# The declared Spark type must match the Python return value.
powerUdf = udf(power, LongType())
# Registration gives SQL a name; DataFrame calls do not require registration.
spark.udf.register("square_value", powerUdf)


In [ ]:
# Use a distinct name to avoid shadowing Spark's built-in power(base, exponent).
spark.sql("SELECT square_value(5) AS squared, square_value(NULL) AS null_squared").show()


## Create a small order dataset
The explicit schema fixes the expected column types. The records are in memory, so these examples do not require Hive or HDFS.


In [ ]:
# discount and taxp are percentages: 5 means 5%, not 0.05.
orders = [
    (1, "iPhone", 100, 1000, 2, 5, 18),
    (2, "Galaxy", 200, 800, 1, 8, 22),
]
orderDf = spark.createDataFrame(orders, schema="""
    product_id long, product_name string, brand_id long,
    price long, qty long, discount long, taxp long
""")
orderDf.show()


## Calculate the amount after discount and tax
The formula is **price ? quantity ? (1 ? discount / 100) ? (1 + tax / 100)**. Tax applies after the discount. Expected amounts are **2242.00** for iPhone and **897.92** for Galaxy.

The function propagates missing inputs. It assumes nonnegative prices/quantities and valid percentage ranges; validate those business constraints before using it. `DoubleType` keeps this teaching example simple. Use decimal arithmetic and an explicit rounding policy for financial amounts.

Avoid printing or performing external writes inside UDFs: code runs in Python workers, and task retries or repeated actions can execute it again. Keep the function deterministic and free of side effects.


In [ ]:
def calculateAmount(price, qty, discount, taxp):
    # Missing inputs produce null; they are not silently replaced with zero.
    if any(value is None for value in (price, qty, discount, taxp)):
        return None
    gross = price * qty
    discounted = gross * (1 - discount / 100.0)
    return float(discounted * (1 + taxp / 100.0))

# This direct call runs on the driver, outside Spark's distributed execution.
print(calculateAmount(1000, 2, 5, 18))  # 2242.0


In [ ]:
# Wrap the Python function for use with Spark columns.
calculate = udf(calculateAmount, DoubleType())
# Register a session-scoped SQL name for the same UDF.
spark.udf.register("calculate", calculate)


In [ ]:
# withColumn builds a lazy plan; show triggers computation.
df = orderDf.withColumn("amount", calculate(col("price"), col("qty"), col("discount"), col("taxp")))
df.printSchema()
df.show()


## Call the same UDF from SQL
A temporary view gives SQL a name for the DataFrame. Both SQL and DataFrame expressions can call the registered function. The portable form in a Python notebook is `spark.sql(...).show()`; `%sql` is an environment-specific notebook magic.


In [ ]:
# A temporary view is scoped to this SparkSession; it is not a stored table.
orderDf.createOrReplaceTempView("orders")


In [ ]:
spark.sql('SHOW TABLES').show()


In [ ]:
spark.sql('SELECT * FROM orders').show()


In [ ]:
# Apply the registered Python UDF in a SQL expression.

df = spark.sql("SELECT *, calculate(price, qty, discount, taxp) as amount from orders")
df.printSchema()
df.show()


In [ ]:
spark.sql('SELECT *, calculate(price, qty, discount, taxp) AS amount FROM orders').show()


## Prefer a built-in expression when it expresses the same rule
Spark can optimize built-in expressions without calling a Python function for each row. Ordinary Python UDFs add Python/JVM communication and serialization costs. JVM UDFs avoid that boundary, but they are not automatically faster than every alternative. Measure with representative data; two rows and notebook startup time are not a useful benchmark.

The expression below uses the same formula and null propagation. Compare its output with the UDF. Floating-point values may have tiny representation differences; round only according to the intended business policy.


In [ ]:
native_amount = (F.col("price") * F.col("qty")
                 * (1 - F.col("discount") / F.lit(100.0))
                 * (1 + F.col("taxp") / F.lit(100.0)))
comparison = (orderDf
    .withColumn("udf_amount", calculate("price", "qty", "discount", "taxp"))
    .withColumn("native_amount", native_amount))
comparison.select("product_name", "udf_amount", "native_amount").show()
# Optional: comparison.explain("formatted") to inspect the Python evaluation stage.


## Scalar Pandas UDF: process a batch of values
A scalar Pandas UDF takes Pandas Series and returns a Series with the same length. Arrow transfers batches between Spark and Python; vectorized Series operations can reduce per-row Python overhead. It does not mean the complete Spark DataFrame is collected into one Pandas DataFrame.

This example expresses the same amount formula. Series arithmetic propagates missing values. Use built-in Spark arithmetic for this particular formula; the Pandas UDF demonstrates the interface for more specialized vectorized logic.


In [ ]:
import pandas as pd
import pyarrow as pa
from pyspark.sql.functions import pandas_udf
print("pandas:", pd.__version__, "pyarrow:", pa.__version__)

@pandas_udf("double")
def calculate_pandas(price: pd.Series, qty: pd.Series,
                     discount: pd.Series, taxp: pd.Series) -> pd.Series:
    return (price * qty * (1 - discount / 100.0) * (1 + taxp / 100.0)).astype("float64")

orderDf.withColumn("amount", calculate_pandas("price", "qty", "discount", "taxp")).show()


## Grouped Pandas operation: one product group at a time
`groupBy(...).applyInPandas(...)` calls an ordinary Python function with a Pandas DataFrame for each group and combines the returned frames. It is a grouped Pandas API, distinct from the scalar `@pandas_udf` above. The function may return a different number of rows from its input.

Grouping requires a shuffle. In Spark 3.5, a whole group is loaded into worker memory, so a heavily skewed product group can cause memory failures. Select only required columns before grouping. The return schema must match the returned names and types.


In [ ]:
data = [
    (1, "Apple", 100.0),
    (2, "Apple", 200.0),
    (3, "Orange", 300.0),
    (4, "Orange", 400.0),
    (5, "Mango", 500.0)
]

columns = ["order_id", "product", "amount"]
df = spark.createDataFrame(data, columns)

df.show()


### Declare the grouped result schema
The function returns one row per product with total, average, and maximum sales. Expected results: Apple `(300, 150, 200)`, Orange `(700, 350, 400)`, and Mango `(500, 500, 500)`. Output ordering is not guaranteed unless we sort it.


In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("product", StringType()),
    StructField("total_sales", DoubleType()),
    StructField("avg_sales", DoubleType()),
    StructField("max_sales", DoubleType())
])


In [ ]:
import pandas as pd

def product_stats(pdf: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "product": [pdf["product"].iloc[0]],
        "total_sales": [pdf["amount"].sum(min_count=1)],
        "avg_sales": [pdf["amount"].mean()],
        "max_sales": [pdf["amount"].max()]
    })

result = df.select("product", "amount").groupBy("product").applyInPandas(product_stats, schema)
result.orderBy("product").show()


## Use native aggregation for standard statistics
Sum, average, and maximum already exist in Spark. Native aggregation is the preferred solution for this example; reserve grouped Pandas logic for calculations that need it. `sum(min_count=1)` in the Pandas function preserves an all-missing group's missing total instead of reporting zero. Missing-value conversions between Pandas/Arrow and Spark still deserve explicit checks on real input data.


In [ ]:
df.groupBy("product").agg(
    F.sum("amount").alias("total_sales"),
    F.avg("amount").alias("avg_sales"),
    F.max("amount").alias("max_sales"),
).orderBy("product").show()


## Notes to keep in mind

- Return values must match the declared Spark type; declare null behavior explicitly.
- Spark evaluates transformations lazily. `show`, `collect`, and writes trigger execution; repeated actions can repeat UDF work.
- Use column expressions as UDF arguments; an ordinary Python function call runs immediately on Python values.
- Do not rely on a surrounding SQL condition to protect an unsafe UDF: handle invalid inputs inside the function or validate them upstream.
- Use representative data and inspect the execution plan before making performance claims.

Try adding a missing price, a zero quantity, and a 100% discount. Predict the results and decide which rows should be rejected by a data-quality contract.

References: [SQL UDF registration](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.UDFRegistration.register.html), [scalar Pandas UDFs](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pandas_udf.html), and [grouped applyInPandas](https://spark.apache.org/docs/3.5.6/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.applyInPandas.html).
